# UNLV Colab benchmark worker v1

This notebook is a generation-only worker. It never scores benchmark outputs and never changes curation policy. Connect with `Select Kernel > Colab > New Colab Server`, upload the two transfer archives, and run the cells in order.

In [3]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)

CompletedProcess(args=['nvidia-smi'], returncode=0)

In [ ]:
%pip install --quiet --extra-index-url https://download.pytorch.org/whl/cu124 torch==2.6.0 transformers==5.9.0 peft==0.19.1 bitsandbytes==0.49.2 datasets==3.6.0 accelerate==1.14.0 tokenizers==0.22.2 safetensors==0.7.0 huggingface-hub==1.17.0
%pip uninstall --quiet -y torchvision

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil

DRIVE_ROOT = Path('/content/drive/MyDrive/UNLV-Colab-Worker-v1')
SOURCE_ARCHIVE = DRIVE_ROOT / 'colab_worker_source_v1.zip'
ADAPTER_PART_ROOT = DRIVE_ROOT / 'adapter_parts_4m'
ADAPTER_ARCHIVE = DRIVE_ROOT / 'colab_adapters_seed101_202_v1.tar'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
ADAPTER_PART_ROOT.mkdir(parents=True, exist_ok=True)

uploaded_source = Path('/content') / SOURCE_ARCHIVE.name
if uploaded_source.is_file() and not SOURCE_ARCHIVE.is_file():
    shutil.copy2(uploaded_source, SOURCE_ARCHIVE)
if not SOURCE_ARCHIVE.is_file():
    raise FileNotFoundError(f'Upload {SOURCE_ARCHIVE.name} to the Colab server first')
uploaded_parts = Path('/content/adapter_parts_4m')
part_names = tuple(f'colab_adapters_seed101_202_v1.tar.part{index:03d}' for index in range(379))
for name in part_names:
    source = uploaded_parts / name
    destination = ADAPTER_PART_ROOT / name
    if source.is_file() and not destination.is_file():
        shutil.copy2(source, destination)
    if not destination.is_file():
        raise FileNotFoundError(f'Upload adapter_parts_4m/{name} to the Colab server first')
if not ADAPTER_ARCHIVE.is_file():
    with ADAPTER_ARCHIVE.open('wb') as output:
        for name in part_names:
            with (ADAPTER_PART_ROOT / name).open('rb') as part:
                shutil.copyfileobj(part, output, length=4 * 1024 * 1024)
print('transfer archives: ready')

In [ ]:
import tarfile
from zipfile import ZipFile

WORK_ROOT = Path('/content/unlv_worker')
SOURCE_ROOT = WORK_ROOT / 'Phase-1'
TRAINING_ROOT = WORK_ROOT / 'training'
THIRD_PARTY_ROOT = WORK_ROOT / 'third_party'
HF_ROOT = WORK_ROOT / 'hf'
for path in (SOURCE_ROOT, TRAINING_ROOT, THIRD_PARTY_ROOT, HF_ROOT):
    path.mkdir(parents=True, exist_ok=True)
with ZipFile(SOURCE_ARCHIVE) as bundle:
    for member in bundle.infolist():
        if member.is_dir():
            continue
        destination = SOURCE_ROOT / member.filename.replace('\\', '/')
        destination.parent.mkdir(parents=True, exist_ok=True)
        with bundle.open(member) as source, destination.open('wb') as target:
            shutil.copyfileobj(source, target)
print('worker source: extracted')

In [ ]:
import hashlib
import json

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def sha256_canonical_crlf(path: Path) -> str:
    content = path.read_bytes().replace(b'\r\n', b'\n').replace(b'\r', b'\n')
    return hashlib.sha256(content.replace(b'\n', b'\r\n')).hexdigest()

manifest_path = SOURCE_ROOT / 'protocols' / 'colab_benchmark_worker_v1.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
for relative, expected in manifest['source']['files'].items():
    actual = sha256(SOURCE_ROOT / relative)
    if actual != expected.lower():
        raise RuntimeError(f'source checksum mismatch: {relative}')
if sha256(ADAPTER_ARCHIVE) != manifest['adapter_transfer']['archive_sha256']:
    raise RuntimeError('adapter archive checksum mismatch')
with tarfile.open(ADAPTER_ARCHIVE) as archive:
    archive.extractall(TRAINING_ROOT, filter='data')
for adapter, files in manifest['adapters'].items():
    for name, expected in files.items():
        actual = sha256(TRAINING_ROOT / 'qlora_runs' / adapter / name)
        if actual != expected.lower():
            raise RuntimeError(f'adapter checksum mismatch: {adapter}/{name}')
report = SOURCE_ROOT / 'artifacts' / manifest['input_report']['relative_path']
if sha256(report) != manifest['input_report']['sha256']:
    raise RuntimeError('training input report checksum mismatch')
print('source and adapter checksums: pass')

In [ ]:
from huggingface_hub import snapshot_download

model_spec = manifest['base_model']
MODEL_PATH = Path(snapshot_download(
    repo_id=model_spec['repo_id'],
    revision=model_spec['revision'],
    cache_dir=HF_ROOT / 'hub',
))
benchmark_spec = manifest['benchmarks']['bigcodebench']
BIGCODEBENCH_PATH = Path(snapshot_download(
    repo_id=benchmark_spec['repo_id'],
    repo_type='dataset',
    revision=benchmark_spec['revision'],
    cache_dir=HF_ROOT / 'hub',
))
print('model snapshot:', MODEL_PATH)
print('BigCodeBench snapshot:', BIGCODEBENCH_PATH)

In [ ]:
def checkout_repository(url: str, destination: Path, revision: str) -> None:
    if not (destination / '.git').is_dir():
        subprocess.run(['git', 'clone', url, str(destination)], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', revision], check=True)
    actual_revision = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    if actual_revision != revision:
        raise RuntimeError(f'repository revision mismatch: {destination}')
    status = subprocess.check_output(['git', '-C', str(destination), 'status', '--porcelain'], text=True).strip()
    if status:
        raise RuntimeError(f'repository working tree is dirty: {destination}')

checkout_repository('https://github.com/facebookresearch/CRUXEval.git', THIRD_PARTY_ROOT / 'cruxeval', manifest['benchmarks']['cruxeval']['git_commit'])
checkout_repository('https://github.com/xlang-ai/DS-1000.git', THIRD_PARTY_ROOT / 'DS-1000', manifest['benchmarks']['ds1000']['git_commit'])
benchmark_files = (
    (BIGCODEBENCH_PATH / 'data' / 'v0.1.4-00000-of-00001.parquet', manifest['benchmarks']['bigcodebench']['data_sha256'], sha256),
    (THIRD_PARTY_ROOT / 'cruxeval' / 'data' / 'cruxeval.jsonl', manifest['benchmarks']['cruxeval']['data_sha256'], sha256_canonical_crlf),
    (THIRD_PARTY_ROOT / 'DS-1000' / 'data' / 'ds1000.jsonl.gz', manifest['benchmarks']['ds1000']['data_sha256'], sha256),
)
for path, expected, hash_function in benchmark_files:
    if hash_function(path) != expected:
        raise RuntimeError(f'benchmark checksum mismatch: {path}')
print('benchmark checksums: pass')

In [ ]:
from importlib.metadata import version
import os
import sys
import torch
import transformers
import peft
import bitsandbytes

os.environ['UNLV_TRAINING_OUTPUT_ROOT'] = str(TRAINING_ROOT)
os.environ['UNLV_BENCHMARK_OUTPUT_ROOT'] = str(DRIVE_ROOT / 'results')
os.environ['UNLV_MODEL_SNAPSHOT_PATH'] = str(MODEL_PATH)
os.environ['UNLV_INPUT_REPORT_PATH'] = str(report)
os.environ['UNLV_THIRD_PARTY_ROOT'] = str(THIRD_PARTY_ROOT)
os.environ['UNLV_HF_HUB_ROOT'] = str(HF_ROOT / 'hub')
os.environ['HF_DATASETS_CACHE'] = str(HF_ROOT / 'datasets')
os.environ['BIGCODEBENCH_DATA_ROOT'] = str(BIGCODEBENCH_PATH)
sys.path.insert(0, str(SOURCE_ROOT))
os.chdir(SOURCE_ROOT)
expected = manifest['environment']
package_names = {
    'torch': 'torch',
    'transformers': 'transformers',
    'peft': 'peft',
    'bitsandbytes': 'bitsandbytes',
    'datasets': 'datasets',
    'accelerate': 'accelerate',
    'tokenizers': 'tokenizers',
    'safetensors': 'safetensors',
    'huggingface_hub': 'huggingface-hub',
}
actual = {'python': '.'.join(map(str, sys.version_info[:3]))}
actual.update({key: version(package) for key, package in package_names.items()})
if '.'.join(actual['python'].split('.')[:2]) != '.'.join(expected['python'].split('.')[:2]):
    raise RuntimeError(f"Python minor-version mismatch: {actual['python']}")
for key in package_names:
    observed = actual[key].split('+', 1)[0] if key == 'torch' else actual[key]
    required = expected[key].split('+', 1)[0] if key == 'torch' else expected[key]
    if observed != required:
        raise RuntimeError(f"package mismatch for {key}: {actual[key]} != {expected[key]}")
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable')
print('environment:', actual)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from external_evaluation.official_suite_generator import preflight_suite

for suite in ('bigcodebench', 'cruxeval_input', 'cruxeval_output', 'ds1000'):
    preflight_suite(suite)
print('remote worker preflight: pass')

In [ ]:
from external_evaluation.official_suite_generator import generate_suites

protocol_path = SOURCE_ROOT / 'protocols' / 'code_7m_normal_hard_confirmatory_v1.json'
jobs = (
    ('raw_audited_natural', 101),
    ('raw_audited_natural', 202),
)
for arm, seed in jobs:
    outputs = generate_suites(
        suites=('ds1000',),
        arm=arm,
        seed=seed,
        max_new_tokens=1024,
        batch_size=4,
        max_batch_context_tokens=4096,
        protocol_path=protocol_path,
        input_report_path=report,
    )
    print('completed:', outputs)